# Stock Data Exploratory Analysis

Module: Markets and Data

## Lesson summary

This notebook turns clean price data into exploratory statistics, empirical return distributions, correlations, rolling diagnostics, and visual evidence for later risk and portfolio modeling. Exploratory analysis does not prove a model; it helps decide which questions are worth modeling {cite}`tukey1977eda`.

## Learning objectives

By the end of this lesson, students should be able to:

- audit missing values and usable date ranges across assets;
- compute simple and log returns from an adjusted price panel;
- summarize count, mean, median, volatility, minimum, maximum, percentiles, skewness, kurtosis, and missingness;
- identify heavy tails, asymmetry, extreme observations, and rolling changes in return behavior;
- interpret correlations, scatterplots, heatmaps, and rolling relationships without treating correlation as causation;
- read plots as financial evidence rather than isolated graphics;
- prepare an EDA table that can be reused in time-series, risk, and portfolio modules.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.market_data import returns_from_prices, synthetic_price_panel
from src.market_data_quality import annualized_volatility, data_quality_report, hampel_outlier_flags

pd.set_option("display.max_columns", 80)

## Price panel

In [ ]:
prices = synthetic_price_panel(periods=520)
prices.tail()

In [ ]:
data_quality_report(prices)

## Returns

Simple returns measure proportional change:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}}.
$$

Log returns are additive across time:

$$
r_t = \ln(P_t) - \ln(P_{t-1}).
$$

In [ ]:
simple_returns = returns_from_prices(prices, method="simple")
log_returns = returns_from_prices(prices, method="log")

log_returns.head()

## Return construction and panel design

Most financial analysis does not use raw prices directly. Prices must often be transformed into returns, aligned across instruments, adjusted for missing values, and organized into panels.

The formulas are simple, but the workflow is fragile. A small mistake in calendars, adjusted prices, missing data, cash distributions, or alignment can produce misleading results.

### Price return and total return

Price return measures the return from price change only. Total return includes cash distributions such as dividends, coupons, or FIBRA distributions:

$$
R_t^{total} = \frac{P_t + C_t}{P_{t-1}} - 1.
$$

where \(C_t\) represents cash distributions during the period.

For equities, adjusted close prices may incorporate dividends and splits depending on the provider. For bonds, FIBRAs, and income-oriented instruments, distributions, accrued interest, and provider methodology require special care.

| Concept | Meaning |
| --- | --- |
| Price return | Return from price change only |
| Total return | Return including price change and cash distributions |
| Nominal return | Return before inflation adjustment |
| Real return | Return after inflation adjustment |
| Local-currency return | Return measured in domestic currency |
| Foreign-currency return | Return measured in another currency |

### Aligning assets

A multi-asset panel requires a common date index. Common choices include:

| Alignment choice | Meaning | Main tradeoff |
| --- | --- | --- |
| inner join | keep only dates available for all series | clean matrix, but may discard useful observations |
| outer join | keep all dates and allow missing values | preserves information, but requires missing-data policy |
| left join | use one reference calendar | useful when one asset or market is the analysis anchor |
| business-day calendar | create a standard calendar and align all assets | explicit, but may create artificial missingness |
| month-end alignment | convert daily series to monthly observations | useful for macro joins, but loses daily detail |

There is no universally correct choice. The correct choice depends on the analysis.

### Wide and long formats

A **wide format** stores one date per row and one asset per column. It is convenient for correlations, covariance matrices, and vectorized return calculations.

A **long format** stores one observation per row. It is often better for storage, metadata, dashboards, and joins with instrument attributes. Tidy data principles help keep variables, observations, and observational units organized {cite}`wickham2014tidy`.

### Financial panel schema

A simple analysis-ready panel should include:

| Column | Description |
| --- | --- |
| `date` | Observation date |
| `ticker` | Instrument identifier |
| `asset_class` | Equity, bond, ETF, FIBRA, FX, macro |
| `price` | Raw price or level |
| `adjusted_price` | Adjusted price, when available |
| `return_simple` | Simple return |
| `return_log` | Log return |
| `currency` | MXN, USD, UDI, etc. |
| `source` | Banxico, INEGI, BMV, yfinance, instructor sample |
| `quality_flag` | Missing, stale, outlier, revised, or reviewed |

### Return checklist

Before calculating returns, confirm:

```text
1. Is the series a price, index, rate, yield, or macro level?
2. Are dates sorted?
3. Are duplicated dates removed or resolved?
4. Are prices adjusted or unadjusted?
5. Are missing prices handled?
6. Is the frequency defined?
7. Is the currency documented?
8. Are corporate actions considered?
9. Is the return formula appropriate?
10. Are results checked for extreme values?
```

## Summary statistics

Descriptive statistics provide a first map of the data. They do not explain the full behavior of a market, but they help identify scale, dispersion, asymmetry, extreme observations, and potential data problems. Financial return distributions often display heavy tails, asymmetry, volatility clustering, and departures from normality {cite}`cont2001empirical`.

Daily volatility is often annualized with the square-root-of-time rule:

$$
\sigma_{ann} \approx \sigma_{daily}\sqrt{252}.
$$

This is a convention, not a universal law. It can be misleading when returns are autocorrelated, volatility changes over time, or the sampling frequency is inconsistent {cite}`lo2002sharpe`.

In [ ]:
summary = pd.DataFrame(
    {
        "valid_observations": log_returns.count(),
        "mean_daily_return": log_returns.mean(),
        "median_daily_return": log_returns.median(),
        "annualized_volatility": annualized_volatility(log_returns),
        "minimum_return": log_returns.min(),
        "maximum_return": log_returns.max(),
        "p01": log_returns.quantile(0.01),
        "skewness": log_returns.skew(),
        "excess_kurtosis": log_returns.kurtosis(),
        "p05": log_returns.quantile(0.05),
        "p25": log_returns.quantile(0.25),
        "p50": log_returns.quantile(0.50),
        "p75": log_returns.quantile(0.75),
        "p95": log_returns.quantile(0.95),
        "p99": log_returns.quantile(0.99),
        "negative_return_share": (log_returns < 0).mean(),
    }
)
summary.sort_values("annualized_volatility", ascending=False)

The mean and median should be compared. A large difference between them may suggest asymmetry, outliers, or a distribution that is not centered in a simple way. Percentiles are especially useful because they summarize downside and upside behavior without assuming a normal distribution {cite}`mcneil2015quantitative`.

Skewness describes asymmetry. Negative skewness means large downside returns are more prominent than large upside returns.

<img src="../../img/generated/eda-skewness-mean-median-mode.png" alt="Skewness diagram comparing mean, median, mode, and asymmetric tails" style="height: 350px; width:850px;"/>

Kurtosis describes tail thickness. High excess kurtosis warns that extreme returns occur more often than a normal approximation would suggest.

<img src="../../img/generated/eda-kurtosis-tail-risk.png" alt="Kurtosis diagram showing tail thickness and extreme-return risk" style="height: 350px; width:850px;"/>

## Rolling descriptive diagnostics

Financial markets change over time. A single mean or volatility for the full sample can hide regimes. Rolling statistics help students see whether return behavior is stable.

In [ ]:
rolling_window = 63
rolling_snapshot = pd.DataFrame(
    {
        "rolling_mean": log_returns.rolling(rolling_window).mean().iloc[-1],
        "rolling_volatility_ann": log_returns.rolling(rolling_window).std().iloc[-1] * (252 ** 0.5),
        "rolling_min": log_returns.rolling(rolling_window).min().iloc[-1],
        "rolling_max": log_returns.rolling(rolling_window).max().iloc[-1],
    }
)
rolling_snapshot

Rolling diagnostics depend on the selected window. A 20-day window, 63-day window, and 252-day window may tell different stories.

## Correlation and covariance

Correlation is an input to diversification, portfolio optimization, and multi-asset risk. It should be interpreted together with the calendar, frequency, and outlier policy that produced the return matrix.

Correlation measures linear association, not causality, stability, or complete dependence {cite}`mcneil2015quantitative`. Two assets may be correlated because they share the same macro driver, sector, currency, interest-rate exposure, risk sentiment, or crisis period.

<img src="../../img/generated/eda-correlation-gallery.png" alt="Correlation gallery showing positive, negative, and weak relationships" style="height: 350px; width:850px;"/>

In [ ]:
correlations = log_returns.corr()
correlations

In [ ]:
covariance = log_returns.cov()
covariance

## Rolling correlation

Correlations can change over time. Relationships that look weak in calm periods may strengthen during stress periods.

In [ ]:
asset_a, asset_b = log_returns.columns[:2]
rolling_correlation = log_returns[asset_a].rolling(rolling_window).corr(log_returns[asset_b])
rolling_correlation.dropna().tail()

Low correlation does not automatically mean low joint risk. Linear correlation can miss nonlinear relationships, asymmetric dependence, and tail dependence.

## Visual review

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

prices.plot(ax=axes[0, 0], title="Price levels")
log_returns.plot(ax=axes[0, 1], title="Daily log returns", alpha=0.75)
log_returns.hist(ax=axes[1, 0], bins=40)
sns.heatmap(correlations, annot=True, cmap="vlag", center=0, ax=axes[1, 1])
axes[1, 1].set_title("Return correlation")

plt.tight_layout()

## Visualization checklist

A financial visualization should answer a clear question.

```text
1. What question does the chart answer?
2. What variable is on each axis?
3. Is the frequency clear?
4. Are units clear?
5. Is the time period visible?
6. Are missing values or breaks visible?
7. Are extreme values explained or flagged?
8. Is the chart comparing levels, returns, or cumulative returns?
9. Are annotations used only when they add context?
10. Can a non-technical reader understand the main point?
```

Visualization should support interpretation, not replace it {cite}`cleveland1993visualizing,wilke2019dataviz`.

## Outlier flags

Large returns are not automatically errors. A flag identifies observations that require review before modeling.

In [ ]:
outlier_counts = pd.Series(
    {
        asset: int(hampel_outlier_flags(log_returns[asset], window=21, n_sigmas=3.0).sum())
        for asset in log_returns.columns
    },
    name="flagged_observations",
)
outlier_counts

## Interpretation discipline

Descriptive statistics are not conclusions by themselves. They are prompts for interpretation.

| Observation | Better question |
| --- | --- |
| High volatility | Was the asset structurally risky or was there a crisis period? |
| Negative skewness | Are losses larger or more abrupt than gains? |
| High kurtosis | Are extreme returns frequent? |
| Low mean return | Is the sample period unfavorable or is the instrument low-return by design? |
| Many missing values | Is the asset illiquid or is the source incomplete? |
| High correlation | Is there a shared macro driver, sector exposure, currency effect, or crisis regime? |

## EDA handoff

| Evidence | Downstream use |
| --- | --- |
| missingness report | determines whether calendar alignment is safe |
| return summary | informs annualization, volatility, and tail-risk assumptions |
| correlation matrix | supports covariance, beta, and diversification analysis |
| outlier flags | separates data audit from model calibration |
| source inventory | documents whether the analysis is reproducible and classroom-safe |